# 📄 Aula 06 — Chat com PDF

Pipeline RAG usando LangChain

**Fluxo:**
1. Carregar PDF
2. Quebrar em chunks
3. Gerar embeddings
4. Guardar no banco vetorial (Chroma)
5. Retriever busca os trechos relevantes
6. LLM responde com base no contexto

In [ ]:
# Instalação das bibliotecas
!pip install -q \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-core \
    chromadb \
    sentence-transformers \
    transformers \
    pypdf \
    huggingface_hub \
    accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6

In [ ]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from transformers import pipeline

print("Imports OK!")

Imports OK!


In [ ]:
# 1. Carregar PDF
# Faça upload do seu PDF no Colab antes de executar esta célula
arquivo = '/content/pdf_1772.pdf'

loader = PyPDFLoader(arquivo)
documents = loader.load()



In [ ]:
print(f"Total de páginas carregadas: {len(documents)}")


Total de páginas carregadas: 5


In [ ]:
print("\nPrimeira página (trecho):")



Primeira página (trecho):


In [ ]:
print(documents[0].page_content[200:2000])

atégica sobre a gestão de pessoas, a nossa especialização em Gestão de Pessoas e Psicologia Organizacional e do
Trabalho é a oportunidade que você estava esperando. 
Com uma formação focada em resultados práticos e soluções inovadoras, este curso é ideal para quem quer se destacar no
mercado de trabalho, seja em grandes empresas, startups ou no campo do empreendedorismo.
Destinado a profissionais de diversas áreas, como psicólogos, gestores, pedagogos e administradores, o curso oferece uma visão
abrangente sobre o comportamento organizacional, a cultura corporativa e as ferramentas mais avançadas para otimizar a gestão de
pessoas. Ao longo da especialização, você aprenderá a aplicar o People Analytics para analisar dados comportamentais e de
desempenho, criando soluções mais eficazes e personalizadas para as necessidades da organização e de seus colaboradores.
A formação também foca em temas essenciais da psicologia no contexto organizacional, como a importância do psicólogo como
Busin

In [ ]:
# 2. Quebrar texto em chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=80
)
docs = text_splitter.split_documents(documents)

print(f"Total de chunks gerados: {len(docs)}")

Total de chunks gerados: 57


In [ ]:
# 3. Embeddings + Banco vetorial + Retriever
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(docs, embeddings)

retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3}
)

print("Banco vetorial e retriever prontos!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Banco vetorial e retriever prontos!


In [ ]:
# 4. LLM local

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200,
)

llm = HuggingFacePipeline(pipeline=pipe)

print("Modelo LLM carregado!")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

Modelo LLM carregado!


In [ ]:
# 5. Prompt + Chain

prompt = PromptTemplate.from_template("""\
Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
{context}

Pergunta: {question}

Resposta:""")

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Pipeline LCEL: recupera → formata → prompt → llm → texto
chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Ok, prompt feito !")

Ok, prompt feito !


In [ ]:
# 6. Loop de perguntas
print("Chat com PDF iniciado! Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Pergunta: ")
    if pergunta.lower() == "sair":
        print("Encerrando...")
        break

    resposta = chain.invoke(pergunta)
    print("\nResposta:", resposta)
    print("-" * 60)

Chat com PDF iniciado! Digite 'sair' para encerrar.

Pergunta: sair
Encerrando...


---
## Groq (gratuito)

1. Crie sua chave gratuita em: https://console.groq.com
2. E teste

In [ ]:
# Testando o GROK
!pip install -q langchain-groq

import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Cole aqui sua chave Groq
os.environ["GROQ_API_KEY"] = ""

llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

prompt = PromptTemplate.from_template("""\
Você é um assistente especializado em análise de documentos.
Responda de forma clara e objetiva com base apenas no contexto abaixo.
Se não encontrar a resposta, diga: "Não encontrei essa informação no documento."

Contexto:
{context}

Pergunta: {question}

Resposta:""")

def formatar_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | prompt
    | llm_groq
    | StrOutputParser()
)

print("Chain com Groq pronta! Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Pergunta: ")
    if pergunta.lower() == "sair":
        print("Encerrando...")
        break

    resposta = chain.invoke(pergunta)
    print("\nResposta:", resposta)
    print("-" * 60)